# BT4012: Two candidates prepared for the next submission day

**Student number: E1156285**

This notebook runs the two new candidates independently of the longer working journal. Its model cells are also included in `e1156285.ipynb`. It reads the supplied competition data, validates both models chronologically and creates two checked CSVs. Run it from the Kaggle project folder using the project's `.venv` kernel.

The competition metric is **ROC-AUC**. Public results for these two candidates are pending. This notebook does not upload anything to Kaggle.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "Data Set"
RANDOM_SEED = 42
FEATURE_COLUMNS = [f"feat_{i}" for i in range(1, 166)]
train = pd.read_csv(DATA_DIR / "Train Data.csv")
test = pd.read_csv(DATA_DIR / "Test Data.csv")
sample_submission = pd.read_csv(DATA_DIR / "Sample Submission" / "BT4012 Competition Submission.csv")
labels = pd.to_numeric(train["label"], errors="coerce")
assert labels.dropna().isin([0, 1]).all()
labeled = train.loc[labels.notna()].copy()
labeled["label"] = labels.loc[labels.notna()].astype(int)
assert train["txId"].is_unique and test["txId"].is_unique
assert set(train["txId"]).isdisjoint(set(test["txId"]))
assert train["time_step"].max() < test["time_step"].min()
assert np.isfinite(train[FEATURE_COLUMNS].to_numpy()).all()
assert np.isfinite(test[FEATURE_COLUMNS].to_numpy()).all()

# Reproduce the existing bin ensemble as a fair reference on these same checks.
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits
reference_records = []
for train_end, valid_start, valid_end in [(14, 15, 28), (21, 22, 35), (28, 29, 35)]:
    learning = labeled.loc[labeled["time_step"] <= train_end]
    checking = labeled.loc[labeled["time_step"].between(valid_start, valid_end)]
    predictions = []
    for bins in (63, 127):
        reference = HistGradientBoostingClassifier(
            max_iter=400, learning_rate=0.05, max_leaf_nodes=15,
            min_samples_leaf=30, l2_regularization=10, max_bins=bins,
            early_stopping=True, validation_fraction=0.1, random_state=42,
        )
        with threadpool_limits(limits=4):
            reference.fit(learning[FEATURE_COLUMNS], learning["label"])
            predictions.append(reference.predict_proba(checking[FEATURE_COLUMNS])[:, 1])
    reference_records.append({"ensemble_auc": roc_auc_score(checking["label"], np.mean(predictions, axis=0))})
validation_comparison = pd.DataFrame(reference_records)
print("Reference ensemble reproduced; known training labels:", len(labeled))


Reference ensemble reproduced; known training labels: 31235


## 19. Prepare two new candidates for the next submission day

We now prepare **XGBoost** and **LightGBM**. Both combine many small decision trees, like our current Histogram Gradient Boosting model. They build and combine trees differently, so we compare their actual predictions rather than assume that changing libraries will help.

### How did we choose these versions?

We tested a bounded set of settings: numeric bins of 31, 63 or 127; different tree limits; stronger regularisation; natural class weights, doubled illicit weights and inverse-frequency weights; and feature ranks within each time step. We also compared internal stopping based on AUC with stopping based on probability loss.

The selected XGBoost and LightGBM versions had the highest equal-weight mean ROC-AUC within their respective families on the **two fourteen-step checks**. We retained their original 165 numeric features and natural class weights. The extra feature ranks and automatic class balancing did not provide a stronger overall result in the tested versions.

These are selected settings from a finite search, not a claim that no better settings exist. The validation periods already influenced our choices, so their scores may be optimistic. The test labels remain hidden.

### What data are we using?

- **Learning and development:** transactions with known labels from steps 1–35.
- **Final predictions:** all 15,329 supplied test transactions from steps 36–49.
- **Input:** the 165 numeric feature columns. Transaction IDs are used to match rows, not to predict fraud.

Unknown labels remain unknown and are excluded from supervised learning. We do not use external transaction labels.

### How do we choose the number of trees?

For each forward check, we reserve the **last three training time steps** as an internal stopping check. The model initially learns from earlier steps and monitors AUC on those three later training steps. It stops if the internal AUC has not improved for 60 rounds, with an upper limit of 400 rounds.

We then start a fresh model and refit it on **all known labels within that training period**, using the chosen number of rounds. Only after this refit do we score the separate, later validation period.

| Forward check | Initial fitting | Internal stopping check | Refit before evaluation | Outer validation |
|---|---|---|---|---|
| Earlier fourteen-step check | 1–11 | 12–14 | 1–14 | 15–28 |
| Later fourteen-step check | 1–18 | 19–21 | 1–21 | 22–35 |
| Short recent check | 1–25 | 26–28 | 1–28 | 29–35 |
| Final test prediction | 1–32 | 33–35 | 1–35 | 36–49, hidden answers |

The outer validation answers never control early stopping. We use AUC to choose when to stop and compare models; the models still learn using a binary classification loss. Setting `eval_metric="auc"` does not directly change that training objective into AUC.

### Packages

XGBoost and LightGBM have been installed in this project's `.venv`, and their versions are recorded in `requirements.txt`. On a new Mac setup, their OpenMP dependency may also require `brew install libomp`. It is already installed on this machine. Use the project's `.venv` kernel in VS Code.


In [2]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from threadpoolctl import threadpool_limits

NEW_FORWARD_CHECKS = [(14, 15, 28), (21, 22, 35), (28, 29, 35)]
NEW_EARLY_STOPPING_ROUNDS = 60
NEW_MAXIMUM_ROUNDS = 400

def fit_with_chronological_stopping(factory, family, learning):
    # The stopping examples belong to the training period, before outer validation.
    training_end = int(learning["time_step"].max())
    fitting = learning.loc[learning["time_step"] <= training_end - 3]
    stopping = learning.loc[learning["time_step"] > training_end - 3]
    assert fitting["time_step"].max() < stopping["time_step"].min()
    assert fitting["label"].nunique() == stopping["label"].nunique() == 2

    initial = factory(NEW_MAXIMUM_ROUNDS, stopping=True)
    with threadpool_limits(limits=4):
        if family == "xgboost":
            initial.fit(
                fitting[FEATURE_COLUMNS], fitting["label"],
                eval_set=[(stopping[FEATURE_COLUMNS], stopping["label"])],
                verbose=False,
            )
            rounds = int(initial.best_iteration + 1)  # XGBoost starts counting at zero.
        else:
            initial.fit(
                fitting[FEATURE_COLUMNS], fitting["label"],
                eval_X=stopping[FEATURE_COLUMNS], eval_y=stopping["label"],
                callbacks=[early_stopping(NEW_EARLY_STOPPING_ROUNDS, verbose=False)],
            )
            rounds = int(initial.best_iteration_)

        assert 1 <= rounds <= NEW_MAXIMUM_ROUNDS
        refitted = factory(rounds, stopping=False)
        refitted.fit(learning[FEATURE_COLUMNS], learning["label"])
    return refitted, rounds

def evaluate_new_candidate(factory, family, display_name):
    records = []
    probabilities_by_period = {}
    for train_end, valid_start, valid_end in NEW_FORWARD_CHECKS:
        learning = labeled.loc[labeled["time_step"] <= train_end]
        checking = labeled.loc[labeled["time_step"].between(valid_start, valid_end)]
        assert learning["time_step"].max() < checking["time_step"].min()
        fitted, rounds = fit_with_chronological_stopping(factory, family, learning)
        with threadpool_limits(limits=4):
            illicit_column = list(fitted.classes_).index(1)
            probabilities = fitted.predict_proba(checking[FEATURE_COLUMNS])[:, illicit_column]
        probabilities_by_period[train_end] = probabilities
        decisions = (probabilities >= 0.5).astype(int)
        records.append({
            "model": display_name,
            "training_steps": f"1–{train_end}",
            "validation_steps": f"{valid_start}–{valid_end}",
            "training_rounds": rounds,
            "validation_auc": roc_auc_score(checking["label"], probabilities),
            "precision_at_0.5": precision_score(checking["label"], decisions, zero_division=0),
            "recall_at_0.5": recall_score(checking["label"], decisions, zero_division=0),
            "f1_at_0.5": f1_score(checking["label"], decisions, zero_division=0),
        })
        print(f"{display_name}; train 1–{train_end}; validate {valid_start}–{valid_end}; "
              f"rounds {rounds}; ROC-AUC {records[-1]['validation_auc']:.4f}", flush=True)
    return pd.DataFrame(records), probabilities_by_period

def save_new_submission(probabilities, filename):
    assert test["index"].is_unique and sample_submission["index"].is_unique
    assert len(test) == len(sample_submission)
    assert set(test["index"]) == set(sample_submission["index"])
    by_identifier = pd.Series(probabilities, index=test["index"])
    candidate = sample_submission[["index"]].copy()
    candidate["target"] = candidate["index"].map(by_identifier)
    assert candidate.columns.tolist() == ["index", "target"]
    assert candidate["index"].equals(sample_submission["index"])
    assert np.isfinite(candidate["target"].to_numpy()).all()
    assert candidate["target"].between(0, 1).all()
    output_directory = PROJECT_DIR / "submissions"
    output_directory.mkdir(exist_ok=True)
    path = output_directory / filename
    candidate.to_csv(path, index=False)
    print("Submission rows:", len(candidate))
    print("Saved to:", path)
    return path


## 20. New model 1: XGBoost with limited leaves and chronological stopping

### What does it do?

XGBoost builds decision trees one after another. Each new tree adjusts the predictions made so far. A tree combines questions about several numeric features, allowing it to recognise patterns that a simple linear model may miss.

We use its **loss-guide** tree-building option: it expands the branch offering the most improvement, while allowing at most **15 leaves** in a tree. A leaf is a final branch where the model makes a prediction adjustment.

### Why this version?

Our earlier XGBoost trials transferred poorly to the harder later period. Limiting the number of leaves, grouping nearby values into fewer bins and using an internal chronological stopping check gave a stronger result in the tested settings.

This selected version scored **0.9685 on steps 15–28** and **0.9153 on steps 22–35**. Our existing ensemble scored **0.9571 and 0.9150**. XGBoost therefore improved both longer checks, but its harder-period gain is only about **0.0003**. That small difference does not prove superiority on the hidden test set.

### What do the settings mean?

- **127 bins:** group nearby numeric values, reducing the number of possible split points.
- **15 leaves:** limit how much detail each tree can learn.
- **Learning rate 0.05:** make small adjustments rather than large jumps.
- **L2 penalty 10:** discourage extreme prediction adjustments.
- **Minimum child weight 3:** require enough statistical support before creating a leaf. This is a sum of curvature weights, not a minimum of three transactions.
- **Natural class weights:** each known training example has weight one. Automatic balancing was tested but was not retained for this version.
- **Up to 400 rounds, patience 60:** choose training length using the earlier internal check described in Section 19.

We use all supplied feature columns, all available known labels in each refit, CPU execution and a fixed random seed. No graph embeddings, guessed labels or transaction IDs are added.

### What will the code produce?

The code checks three future periods, displays AUC and the additional classification metrics, then trains a fresh final model using steps 1–35. It predicts probabilities for steps 36–49 and saves **`submission_xgboost_tuned.csv`**.

The 0.5 threshold is used only to explain precision, recall and F1. The CSV keeps the original probability scores for Kaggle's ROC-AUC evaluation.

Technical reference: [XGBoost parameters](https://xgboost.readthedocs.io/en/stable/parameter.html).


In [3]:
def make_selected_xgboost(rounds, *, stopping=False):
    return XGBClassifier(
        n_estimators=int(rounds),
        learning_rate=0.05,
        tree_method="hist",
        grow_policy="lossguide",
        max_depth=0,             # Overall complexity is limited by the leaf count.
        max_leaves=15,
        max_bin=127,
        min_child_weight=3,
        reg_lambda=10,
        subsample=1.0,
        colsample_bytree=1.0,
        scale_pos_weight=1.0,
        objective="binary:logistic",
        eval_metric="auc",
        early_stopping_rounds=NEW_EARLY_STOPPING_ROUNDS if stopping else None,
        n_jobs=4,
        random_state=RANDOM_SEED,
    )

xgboost_validation, xgboost_validation_probabilities = evaluate_new_candidate(
    make_selected_xgboost, "xgboost", "XGBoost, 127 bins / 15 leaves"
)
display(xgboost_validation.round(4))

xgboost_final_model, xgboost_final_rounds = fit_with_chronological_stopping(
    make_selected_xgboost, "xgboost", labeled
)
with threadpool_limits(limits=4):
    illicit_column = list(xgboost_final_model.classes_).index(1)
    xgboost_test_probabilities = xgboost_final_model.predict_proba(
        test[FEATURE_COLUMNS]
    )[:, illicit_column]
xgboost_submission_path = save_new_submission(
    xgboost_test_probabilities, "submission_xgboost_tuned.csv"
)
print("Final training rounds:", xgboost_final_rounds)

new_model_directory = PROJECT_DIR / "models"
new_model_directory.mkdir(exist_ok=True)
xgboost_final_model.save_model(new_model_directory / "xgboost_tuned.json")
xgboost_validation.to_csv(PROJECT_DIR / "submissions" / "xgboost_tuned_validation.csv", index=False)


XGBoost, 127 bins / 15 leaves; train 1–14; validate 15–28; rounds 185; ROC-AUC 0.9685


XGBoost, 127 bins / 15 leaves; train 1–21; validate 22–35; rounds 381; ROC-AUC 0.9153


XGBoost, 127 bins / 15 leaves; train 1–28; validate 29–35; rounds 398; ROC-AUC 0.9958


,model,training_steps,validation_steps,training_rounds,validation_auc,precision_at_0.5,recall_at_0.5,f1_at_0.5
0,"XGBoost, 127 bins / 15 leaves",1–14,15–28,185,0.9685,0.9770,0.8022,0.8810
1,"XGBoost, 127 bins / 15 leaves",1–21,22–35,381,0.9153,0.9291,0.6723,0.7801
2,"XGBoost, 127 bins / 15 leaves",1–28,29–35,398,0.9958,0.9770,0.8485,0.9082


Submission rows: 15329
Saved to: /Users/mlvlo/BT4012/Kaggle/submissions/submission_xgboost_tuned.csv
Final training rounds: 229


## 21. New model 2: LightGBM with coarse bins and chronological stopping

### What does it do?

LightGBM also combines a sequence of decision trees. It grows the branch offering the greatest improvement and groups numeric values into bins to make training efficient.

Our chosen XGBoost version also uses this general leaf-growing idea. The libraries differ in their binning and split calculations, and in how they limit small branches. We compare their results rather than assume that one implementation must be better.

### Why this version?

We use only **31 bins**, creating coarser groups of numeric values than our XGBoost candidate. This deliberately gives the model fewer opportunities to learn small numerical details that might fail to transfer to future periods.

The selected LightGBM version scored **0.9718 on steps 15–28** and **0.8962 on steps 22–35**. Its equal-weight mean over those checks is approximately **0.9340**. This was the strongest mean among the LightGBM versions tested, but our current ensemble's mean was **0.9360** and XGBoost's was **0.9419**.

LightGBM is therefore an **alternative submission candidate**. Its stronger earlier-period result did not carry over to the harder later period. We cannot yet call it an improvement over our best submitted model.

### What do the settings mean?

- **31 bins:** use coarser numeric groupings to test whether they transfer better.
- **15 leaves:** keep each tree's overall size limited.
- **At least 30 transactions per leaf:** reduce reliance on very small groups.
- **Learning rate 0.05 and L2 penalty 10:** make cautious adjustments and discourage extreme ones.
- **Natural class weights:** do not automatically multiply the illicit class's influence. Weighted alternatives were tested.
- **Up to 400 rounds, patience 60:** select the training length using an internal check from the training period, then refit before scoring later transactions.

The `deterministic` option and fixed random seed support repeatability. Trees can combine features without requiring the scaling step used by logistic regression.

### What will the code produce?

The code checks the same three chronological periods as XGBoost, trains a fresh final model and saves **`submission_lightgbm_tuned.csv`**, containing the predicted illicit probability for each test transaction.

Its public and private Kaggle scores are still unknown. The CSV is ready for a comparison submission, with the local limitations recorded honestly.

Technical reference: [LightGBM classifier documentation](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMClassifier.html).


In [4]:
def make_selected_lightgbm(rounds, *, stopping=False):
    return LGBMClassifier(
        n_estimators=int(rounds),
        learning_rate=0.05,
        num_leaves=15,
        max_depth=-1,
        max_bin=31,
        min_child_samples=30,
        reg_lambda=10,
        subsample=1.0,
        subsample_freq=0,
        colsample_bytree=1.0,
        scale_pos_weight=1.0,
        objective="binary",
        metric="auc",
        deterministic=True,
        force_col_wise=True,
        verbosity=-1,
        n_jobs=4,
        random_state=RANDOM_SEED,
    )

lightgbm_validation, lightgbm_validation_probabilities = evaluate_new_candidate(
    make_selected_lightgbm, "lightgbm", "LightGBM, 31 bins / 15 leaves"
)
display(lightgbm_validation.round(4))

lightgbm_final_model, lightgbm_final_rounds = fit_with_chronological_stopping(
    make_selected_lightgbm, "lightgbm", labeled
)
with threadpool_limits(limits=4):
    illicit_column = list(lightgbm_final_model.classes_).index(1)
    lightgbm_test_probabilities = lightgbm_final_model.predict_proba(
        test[FEATURE_COLUMNS]
    )[:, illicit_column]
lightgbm_submission_path = save_new_submission(
    lightgbm_test_probabilities, "submission_lightgbm_tuned.csv"
)
print("Final training rounds:", lightgbm_final_rounds)

new_model_directory = PROJECT_DIR / "models"
new_model_directory.mkdir(exist_ok=True)
lightgbm_final_model.booster_.save_model(str(new_model_directory / "lightgbm_tuned.txt"))
lightgbm_validation.to_csv(PROJECT_DIR / "submissions" / "lightgbm_tuned_validation.csv", index=False)


LightGBM, 31 bins / 15 leaves; train 1–14; validate 15–28; rounds 161; ROC-AUC 0.9718


LightGBM, 31 bins / 15 leaves; train 1–21; validate 22–35; rounds 149; ROC-AUC 0.8962


LightGBM, 31 bins / 15 leaves; train 1–28; validate 29–35; rounds 397; ROC-AUC 0.9966


,model,training_steps,validation_steps,training_rounds,validation_auc,precision_at_0.5,recall_at_0.5,f1_at_0.5
0,"LightGBM, 31 bins / 15 leaves",1–14,15–28,161,0.9718,0.9787,0.7476,0.8477
1,"LightGBM, 31 bins / 15 leaves",1–21,22–35,149,0.8962,0.9325,0.6701,0.7798
2,"LightGBM, 31 bins / 15 leaves",1–28,29–35,397,0.9966,0.9758,0.9165,0.9453


Submission rows: 15329
Saved to: /Users/mlvlo/BT4012/Kaggle/submissions/submission_lightgbm_tuned.csv
Final training rounds: 203


## 22. Compare the new candidates and prepare for submission

The table below compares each model on exactly the same labelled transactions. The mean gives equal weight to the two fourteen-step checks. The short recent check provides additional context and was not used to choose these settings.

Our development periods overlap and were used repeatedly during model selection. Treat these as development scores, with uncertainty about future performance.

**Try XGBoost first when the submission allowance is available.** It improved both fourteen-step checks over the current ensemble, although the harder-period improvement is small.

LightGBM is ready as a second comparison candidate. It was weaker on the harder local check, so uploading it is an exploratory comparison rather than an evidence-backed claim that it will beat XGBoost or our current best submission.

The current ensemble's public score remains **0.94199**. The two new public scores are **pending**. Neither candidate has yet demonstrated a Kaggle score above 0.96, and their local scores cannot guarantee it.

The two files to upload are `submission_xgboost_tuned.csv` and `submission_lightgbm_tuned.csv` in the `submissions` folder. Creating them locally does not consume submissions. We have not uploaded them.

We keep the previous ensemble file for comparison. After submitting, record both new public scores and keep the strongest supported candidates available for the final private leaderboard selection.


In [5]:
new_candidate_comparison = pd.DataFrame({
    "validation_steps": xgboost_validation["validation_steps"],
    "current_ensemble_auc": validation_comparison["ensemble_auc"].to_numpy(),
    "xgboost_auc": xgboost_validation["validation_auc"].to_numpy(),
    "lightgbm_auc": lightgbm_validation["validation_auc"].to_numpy(),
})
display(new_candidate_comparison.round(4))
print("Mean ROC-AUC across the two fourteen-step checks:")
display(new_candidate_comparison.iloc[:2].select_dtypes("number").mean().round(4))
new_candidate_comparison.to_csv(
    PROJECT_DIR / "submissions" / "new_booster_comparison.csv", index=False
)

# Verify that the saved models reproduce their original probability predictions.
from lightgbm import Booster
reloaded_xgboost = XGBClassifier(n_jobs=4)
reloaded_xgboost.load_model(new_model_directory / "xgboost_tuned.json")
reloaded_lightgbm = Booster(model_file=str(new_model_directory / "lightgbm_tuned.txt"))
with threadpool_limits(limits=4):
    assert np.allclose(reloaded_xgboost.predict_proba(test[FEATURE_COLUMNS])[:, 1],
                       xgboost_test_probabilities, rtol=0, atol=1e-12)
    assert np.allclose(reloaded_lightgbm.predict(test[FEATURE_COLUMNS]),
                       lightgbm_test_probabilities, rtol=0, atol=1e-12)
print("Both saved models reproduce their probabilities. Both CSVs are ready.")


,validation_steps,current_ensemble_auc,xgboost_auc,lightgbm_auc
0,15–28,0.9571,0.9685,0.9718
1,22–35,0.9150,0.9153,0.8962
2,29–35,0.9951,0.9958,0.9966


Mean ROC-AUC across the two fourteen-step checks:


current_ensemble_auc    0.9360
xgboost_auc             0.9419
lightgbm_auc            0.9340
dtype: float64

Both saved models reproduce their probabilities. Both CSVs are ready.
